# 6주차 실습: 로지스틱 회귀
## AI는 어떻게 확률로 판단할까?

---
**학습 목표**
1. 로지스틱 회귀 모델로 당뇨 위험을 **확률**로 예측한다
2. `predict_proba()`로 확률값을 직접 꺼내 해석한다
3. 임계값(Threshold)을 바꾸며 Precision·Recall 변화를 체험한다
4. Decision Tree·Random Forest와 성능을 비교한다
5. 오즈비(OR)로 각 변수의 영향력을 해석한다
---

> **실습 규칙**: 셀을 **위에서 아래로** 순서대로 실행하세요. 중간을 건너뛰면 오류가 납니다!

---
> 📂 **이 버전은 `week6_실습자료.xlsx` 파일을 직접 사용합니다.**
> Colab 왼쪽 파일 탭에 `week6_실습자료.xlsx`를 업로드한 뒤 실행하세요.

## STEP 0. 준비 — 가장 먼저 실행하세요

⏱ 한글 폰트 설치에 약 30초 소요됩니다.

In [ ]:
import subprocess, warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설치 (Colab 필수)
subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-nanum'], capture_output=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

print('✅ 준비 완료! 다음 셀로 이동하세요.')

## STEP 1. 데이터 불러오기

엑셀 파일(`week6_실습자료.xlsx`)의 **`당뇨_데이터`** 시트를 사용합니다.

| 컬럼 | 설명 | 비고 |
|------|------|------|
| 나이 | 나이 (세) | |
| BMI | 체질량지수 (kg/m²) | **결측값 있음** |
| 혈당수치 | 혈당 (mg/dL) | |
| 혈압 | 혈압 (mmHg) | |
| 운동빈도 | 주간 운동 횟수 | **결측값 있음** |
| 가족력 | 당뇨 가족력 (있음/없음) | **→ 숫자로 변환 필요** |
| 인슐린 | 인슐린 수치 (μU/mL) | |
| 당뇨위험 | 정답 레이블 (1=고위험, 0=저위험) | |

In [ ]:
# ── 엑셀 파일에서 데이터 불러오기 ──────────────────────────────────────────
# Colab 왼쪽 파일 탭에 week6_실습자료.xlsx 를 먼저 업로드하세요!
df = pd.read_excel(
    'week6_실습자료.xlsx',
    sheet_name='당뇨_데이터',
    header=2,       # 3번째 행이 컬럼명
    skiprows=[3]    # 4번째 행(단위 설명행) 건너뜀
)
df.columns = ['나이', 'BMI', '혈당수치', '혈압', '운동빈도', '가족력', '인슐린', '당뇨위험']

print(f'데이터 크기: {df.shape}')   # (200, 8)
print()
print('=== 컬럼별 결측값 ===')
print(df.isnull().sum())
print()
print('=== 가족력 값 종류 ===')
print(df['가족력'].value_counts())
print()
print('=== 당뇨 여부 분포 ===')
print(df['당뇨위험'].value_counts())
print(f'\n고위험(1) 비율: {df["당뇨위험"].mean():.1%}')

df.head(8)

In [ ]:
# 데이터 시각화 — 혈당·BMI 분포
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df[df['당뇨위험']==0]['혈당수치'], bins=25, alpha=0.7, color='steelblue', label='저위험(0)')
axes[0].hist(df[df['당뇨위험']==1]['혈당수치'], bins=25, alpha=0.7, color='tomato',    label='고위험(1)')
axes[0].set_xlabel('혈당수치 (mg/dL)')
axes[0].set_ylabel('환자 수')
axes[0].set_title('저위험 vs 고위험 — 혈당 분포 비교')
axes[0].legend()
axes[0].axvline(126, color='black', linestyle='--', linewidth=1.5, label='당뇨 기준선 (126)')

axes[1].hist(df[df['당뇨위험']==0]['BMI'].dropna(), bins=25, alpha=0.7, color='steelblue', label='저위험(0)')
axes[1].hist(df[df['당뇨위험']==1]['BMI'].dropna(), bins=25, alpha=0.7, color='tomato',    label='고위험(1)')
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('환자 수')
axes[1].set_title('저위험 vs 고위험 — BMI 분포 비교')
axes[1].legend()

plt.tight_layout()
plt.show()

## STEP 2. 전처리 — 스케일링이 핵심!

엑셀 데이터에는 **2가지 추가 처리**가 필요합니다.

| 단계 | 내용 | 이 파일의 특이사항 |
|------|------|-------------------|
| ① 결측값 처리 | BMI·운동빈도 → 평균으로 대체 | BMI 25개, 운동빈도 20개 결측 |
| **② 가족력 인코딩** | **있음→1, 없음→0** | **★ 엑셀 전용 단계** |
| ③ Feature/Label 분리 | X, y | |
| ④ 훈련/테스트 분리 | 80:20 | |
| ⑤ StandardScaler | 평균=0, 표준편차=1 | 로지스틱 회귀 필수 |

In [ ]:
df_clean = df.copy()

# ① 결측값 처리 — BMI, 운동빈도만 결측 존재
missing_cols = ['BMI', '운동빈도']
for col in missing_cols:
    mean_val = df_clean[col].mean()
    df_clean[col] = df_clean[col].fillna(mean_val)
    print(f'{col} 결측값 → 평균({mean_val:.2f})으로 대체')

# ② 가족력 인코딩 ★ 엑셀 전용
#    '있음' → 1,  '없음' → 0  (문자열을 숫자로 변환)
df_clean['가족력'] = df_clean['가족력'].map({'있음': 1, '없음': 0})
print(f'\n가족력 인코딩 완료: 있음→1, 없음→0')
print(f'가족력 분포: {df_clean["가족력"].value_counts().to_dict()}')

# ③ Feature / Label 분리
X = df_clean.drop('당뇨위험', axis=1)
y = df_clean['당뇨위험']
print(f'\nFeature 변수: {X.columns.tolist()}')
print(f'X shape: {X.shape}  |  y shape: {y.shape}')

# ④ 훈련 / 테스트 분리 (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\n훈련 데이터: {X_train.shape[0]}명')
print(f'테스트 데이터: {X_test.shape[0]}명')
print(f'고위험 비율 (훈련): {y_train.mean():.1%}')
print(f'고위험 비율 (테스트): {y_test.mean():.1%}')

# ⑤ StandardScaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # 훈련 데이터로 fit + transform
X_test_s  = scaler.transform(X_test)        # 테스트 데이터는 transform만!

print(f'\n혈당수치 — 스케일링 전: 평균={X_train["혈당수치"].mean():.1f}, 표준편차={X_train["혈당수치"].std():.1f}')
print(f'혈당수치 — 스케일링 후: 평균={X_train_s[:,2].mean():.4f}, 표준편차={X_train_s[:,2].std():.4f}')
print('\n✅ 전처리 완료!')

## STEP 3. 로지스틱 회귀 학습

```python
# 코드 비교: 모델 이름만 바뀌고 나머지는 완전히 동일!
model = DecisionTreeClassifier()       # 4주차
model = RandomForestClassifier()       # 5주차
model = LogisticRegression()           # 오늘!
```

In [ ]:
# 로지스틱 회귀 학습
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

# 예측
y_pred_lr = lr.predict(X_test_s)

print('=== 로지스틱 회귀 성능 ===')
print(f'정확도: {accuracy_score(y_test, y_pred_lr):.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['저위험(0)', '고위험(1)']))

# ※ 참고: 이 결과는 엑셀 200행 기준입니다.
#    강의자료 PPT의 수치(정확도 85.7% 등)는 Pima 768행 원본 기준이므로 다를 수 있습니다.

## STEP 4. ★ predict_proba — 오늘 실습의 핵심!

`predict()` → 0 또는 1만 반환 (기존 방식)

`predict_proba()` → **[저위험 확률, 고위험 확률]** 반환 (로지스틱 회귀의 강점!)

```
[0.13, 0.87]  →  저위험 확률 13%, 고위험(당뇨) 확률 87%
```
임상 보고서의 '위험도 87%'가 바로 이 값입니다.

In [ ]:
# 확률값 추출
y_proba = lr.predict_proba(X_test_s)
diabetes_prob = y_proba[:, 1]  # 고위험(당뇨) 확률만 추출

print('=== 첫 10명의 예측 결과 ===')
print(f'{"번호":>4} | {"저위험확률":>8} | {"고위험확률":>8} | {"예측":>5} | {"실제":>5} | 판정')
print('-' * 55)
for i in range(10):
    p_low      = y_proba[i][0]
    p_high     = y_proba[i][1]
    pred       = y_pred_lr[i]
    actual     = y_test.iloc[i]
    correct    = '✅ 맞음' if pred == actual else '❌ 틀림'
    print(f'{i+1:>4} | {p_low:>7.1%} | {p_high:>7.1%} | {pred:>5} | {actual:>5} | {correct}')

In [ ]:
# 당뇨(고위험) 확률 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 히스토그램
axes[0].hist(diabetes_prob[y_test == 0], bins=20, alpha=0.75, color='steelblue', label='실제 저위험', edgecolor='white')
axes[0].hist(diabetes_prob[y_test == 1], bins=20, alpha=0.75, color='tomato',    label='실제 고위험',  edgecolor='white')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=2, label='임계값 0.5')
axes[0].set_xlabel('고위험(당뇨) 예측 확률')
axes[0].set_ylabel('환자 수')
axes[0].set_title('고위험 예측 확률 분포')
axes[0].legend()
axes[0].set_xlim(0, 1)

# 오른쪽: 산점도
colors = ['tomato' if y == 1 else 'steelblue' for y in y_test]
axes[1].scatter(range(len(diabetes_prob)), diabetes_prob, c=colors, alpha=0.55, s=25)
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1.5, label='임계값 0.5')
axes[1].set_xlabel('환자 번호')
axes[1].set_ylabel('고위험 예측 확률')
axes[1].set_title('환자별 고위험 예측 확률')
axes[1].legend()

plt.tight_layout()
plt.show()
print('파란 점: 실제 저위험  /  빨간 점: 실제 고위험')

## STEP 5. 임계값(Threshold) 실험

**임계값이란?** '이 확률 이상이면 고위험으로 판정!'하는 기준값

| 임계값 | 효과 | 임상 적용 |
|--------|------|-----------|
| 낮춤 (0.3) | Recall↑, Precision↓ | 놓치면 안 될 때 (암 검진) |
| 기본 (0.5) | 균형 | 일반 스크리닝 |
| 높임 (0.7) | Precision↑, Recall↓ | 확실한 경우만 판정 |

> 📋 **엑셀 연동**: 실행 후 나오는 수치를 `임계값_실험결과` 시트에 입력하면 차트가 자동 업데이트됩니다!

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []

print('=== 임계값별 성능 비교 ===')
print(f'{"임계값":>6} | {"정확도":>8} | {"정밀도":>8} | {"재현율":>8} | {"F1":>8}')
print('-' * 50)

for thr in thresholds:
    y_custom = (diabetes_prob >= thr).astype(int)
    acc  = accuracy_score(y_test, y_custom)
    prec = precision_score(y_test, y_custom, zero_division=0)
    rec  = recall_score(y_test, y_custom, zero_division=0)
    f1   = f1_score(y_test, y_custom, zero_division=0)
    results.append({'임계값': thr, '정확도': acc, '정밀도': prec, '재현율': rec, 'F1': f1})
    marker = '  ← 기본값' if thr == 0.5 else ''
    print(f'{thr:>6.1f} | {acc:>8.4f} | {prec:>8.4f} | {rec:>8.4f} | {f1:>8.4f}{marker}')

df_results = pd.DataFrame(results)

print()
print('📋 위 수치를 엑셀 「임계값_실험결과」 시트에 입력하세요!')

In [ ]:
# 임계값에 따른 성능 변화 그래프
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 라인 그래프
axes[0].plot(df_results['임계값'], df_results['정밀도'], 'o-', color='steelblue', linewidth=2, markersize=8, label='정밀도(Precision)')
axes[0].plot(df_results['임계값'], df_results['재현율'], 's-', color='tomato',    linewidth=2, markersize=8, label='재현율(Recall)')
axes[0].plot(df_results['임계값'], df_results['F1'],     '^-', color='forestgreen',linewidth=2, markersize=8, label='F1 Score')
axes[0].axvline(0.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='기본 임계값(0.5)')
axes[0].set_xlabel('임계값 (Threshold)')
axes[0].set_ylabel('성능 지표')
axes[0].set_title('임계값에 따른 Precision · Recall · F1 변화')
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

# 오른쪽: 막대 그래프
x = np.arange(len(thresholds))
w = 0.25
axes[1].bar(x - w, df_results['정밀도'], w, label='정밀도', color='steelblue', alpha=0.85)
axes[1].bar(x,     df_results['재현율'], w, label='재현율', color='tomato',    alpha=0.85)
axes[1].bar(x + w, df_results['F1'],     w, label='F1',     color='forestgreen',alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in thresholds])
axes[1].set_xlabel('임계값')
axes[1].set_title('임계값별 성능 막대 비교')
axes[1].legend()
axes[1].set_ylim(0, 1.1)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print('임계값을 낮추면 Recall↑(놓치지 않음), Precision↓(과잉 판정)')
print('임계값을 높이면 Precision↑(정확한 판정), Recall↓(놓치는 환자 증가)')

## STEP 6. 3모델 최종 성능 비교

Decision Tree · Random Forest · 로지스틱 회귀를 **동일한 데이터**로 비교합니다.

> 📋 **엑셀 연동**: 실행 후 수치를 `3모델_성능비교` 시트에 입력하면 최고 모델이 자동 표시됩니다!

In [ ]:
# 3모델 학습 및 성능 수집
models = {
    'Decision Tree\n(4주차)':       (DecisionTreeClassifier(max_depth=5, random_state=42),  X_train,   X_test),
    'Random Forest\n(5주차)':       (RandomForestClassifier(n_estimators=100, random_state=42), X_train, X_test),
    'Logistic Regression\n(오늘!)': (lr, X_train_s, X_test_s),
}

comparison = []
print('=== 3모델 최종 성능 비교 ===')
print(f'{"모델":<22} | {"정확도":>8} | {"정밀도":>8} | {"재현율":>8} | {"F1":>8}')
print('-' * 65)

for name, (model, X_tr, X_te) in models.items():
    model.fit(X_tr, y_train)
    pred = model.predict(X_te)
    acc  = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec  = recall_score(y_test, pred, zero_division=0)
    f1   = f1_score(y_test, pred, zero_division=0)
    comparison.append({'모델': name.replace('\n', ' '), '정확도': acc, '정밀도': prec, '재현율': rec, 'F1': f1})
    print(f'{name.replace(chr(10)," "):<22} | {acc:>8.4f} | {prec:>8.4f} | {rec:>8.4f} | {f1:>8.4f}')

df_compare = pd.DataFrame(comparison)

print()
print('📋 위 수치를 엑셀 「3모델_성능비교」 시트에 입력하세요!')

In [ ]:
# 3모델 시각화
metrics_list = ['정확도', '정밀도', '재현율', 'F1']
x = np.arange(len(metrics_list))
w = 0.25
model_colors = ['#E07840', '#2D7A60', '#1E8A6E']
model_labels = ['Decision Tree (4주차)', 'Random Forest (5주차)', 'Logistic Regression (오늘!)']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (row, color, label) in enumerate(zip(df_compare.iterrows(), model_colors, model_labels)):
    _, data = row
    vals = [data[m] for m in metrics_list]
    bars = ax.bar(x + (i-1)*w, vals, w, label=label, color=color, alpha=0.88)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_list, fontsize=12)
ax.set_ylim(0, 1.18)
ax.set_ylabel('성능 지표')
ax.set_title('3모델 성능 비교 — Decision Tree vs Random Forest vs 로지스틱 회귀', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\n어떤 모델이 더 낫다기보다, 목적에 따라 적합한 모델이 달라집니다!')
print('  - 해석이 필요할 때(논문·임상) → 로지스틱 회귀')
print('  - 놓치면 안 될 때(암 검진)   → Recall 높은 모델')
print('  - 정확도가 중요할 때          → Random Forest')

## BONUS. 오즈비(Odds Ratio) — 논문에서 바로 읽기

보건의료 논문에서 자주 보는 **OR = 2.5** 의 의미를 직접 계산합니다.

> **로지스틱 회귀 계수(coefficient)를 지수화하면 오즈비가 됩니다.**
>
> `OR = exp(β계수)`

OR > 1 → 위험 증가 요인 | OR < 1 → 보호 요인

> 📋 **엑셀 연동**: 계수(β) 값을 `오즈비_참고표` 시트 **B열**에 입력하면 OR이 자동 계산됩니다!

In [ ]:
feature_names    = X.columns.tolist()
coefficients     = lr.coef_[0]
odds_ratios      = np.exp(coefficients)

df_or = pd.DataFrame({
    '변수':      feature_names,
    '계수(β)':   coefficients,
    '오즈비(OR)': odds_ratios
}).sort_values('오즈비(OR)', ascending=False).reset_index(drop=True)

print('=== 오즈비 (높은 순) ===')
print(f'{"변수":8s} | {"계수(β)":>10} | {"오즈비(OR)":>10} | 해석')
print('-' * 55)
for _, row in df_or.iterrows():
    direction = '위험 증가 ↑' if row['오즈비(OR)'] > 1 else '보호 요인 ↓'
    print(f'{row["변수"]:8s} | {row["계수(β)"]:>+10.4f} | {row["오즈비(OR)"]:>10.4f} | {direction}')

print()
print('📋 위 계수(β) 값을 엑셀 「오즈비_참고표」 시트 B열에 입력하세요!')

In [ ]:
# 오즈비 막대 차트
fig, ax = plt.subplots(figsize=(10, 6))

colors_or = ['tomato' if or_ > 1 else 'steelblue' for or_ in df_or['오즈비(OR)']]
bars = ax.barh(df_or['변수'], df_or['오즈비(OR)'], color=colors_or, alpha=0.85, edgecolor='white')
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='OR=1.0 (영향 없음)')

for bar, val in zip(bars, df_or['오즈비(OR)']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('오즈비 (Odds Ratio)')
ax.set_title('각 변수의 당뇨 위험 오즈비 — 로지스틱 회귀 결과', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('빨간 막대 (OR>1): 해당 변수 증가 시 당뇨 위험 증가')
print('파란 막대 (OR<1): 해당 변수 증가 시 당뇨 위험 감소 (보호 요인)')

---
## 6주차 핵심 정리

| 개념 | 한 줄 요약 |
|------|------------|
| 로지스틱 회귀 | 확률(0~1)로 분류하는 모델 |
| 시그모이드 함수 | 어떤 숫자든 0~1 사이 확률로 변환 |
| StandardScaler | 로지스틱 회귀에 필수 — 스케일 통일 |
| 가족력 인코딩 | 문자열(있음/없음) → 숫자(1/0) 변환 |
| predict_proba() | 0/1이 아닌 **확률값** 반환 |
| 임계값(Threshold) | 어디서부터 양성으로 볼지 기준 — 조정 가능 |
| 오즈비(OR) | 변수가 결과에 미치는 영향의 배수 |

**⚠ 수치 참고 안내**
강의 PPT의 성능 수치(예: 정확도 85.7%, RF 91%)는 Pima 원본 데이터(768명) 기준입니다.
이 실습은 엑셀 데이터(200명)를 사용하므로 수치가 다를 수 있으며, 이는 정상입니다.
중요한 것은 **수치의 크기가 아니라 방향성**(임계값↓ → Recall↑)을 이해하는 것입니다.


오늘 만든 모델을 더 믿을 수 있게 만드는 방법!